# GT Diagnostic Analysis v2

**Purpose:** Diagnose GT quality across two independent record sets

**Scope:**
1. **Golden test set (45 records)** -- final holdout/test; must be validated before any optimisation
2. **New 89 diagnostic set** -- reconstituted working set (DD-2026-006); 63 surviving + 26 replacements

**Approach:**
- Run existing DSPy CoT extraction (unchanged signature) on both sets
- Flag structural GT issues: granularity mismatch, non-standard IDs, supplementary-only data
- Generate separate reports per set + combined comparison
- Golden assessed first (blocking dependency for all downstream experiments)

**Key change from v1:** Parameterised GT dir and XML mapping per set so both can run in a single session

In [41]:
!pip install -q dspy-ai anthropic

In [42]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [43]:
import json
import os
import re
import time
from pathlib import Path
from typing import Dict, List, Any, Optional, Tuple, Set
from datetime import datetime
from collections import defaultdict, Counter
from dataclasses import dataclass, field, asdict
import dspy

print(f"DSPy version: {dspy.__version__}")

DSPy version: 3.1.3


In [44]:
# =============================================================================
# Configuration
# =============================================================================

BASE_DIR = Path("/content/drive/MyDrive/AI6129/")

# --- Original 227 GT and XML (used for the new 89) ---
GT_DIR_227 = BASE_DIR / "ground_truth"
XML_DIR_227 = BASE_DIR / "xml"

# --- Golden GT and XML (separate directory) ---                               #changed
GT_DIR_GOLDEN = BASE_DIR / "ground_truth" / "golden"                           #changed
XML_DIR_GOLDEN = BASE_DIR / "xml" / "golden"                                   #changed

# --- Splits file (v2 with reconstituted 89) ---                               #changed
SPLITS_FILE = BASE_DIR / "assay" / "validation_splits" / "assay_tadp_gepa_optimised_splits_v2.json"  #changed

# --- Output directories (separate per set) ---                                #changed
OUTPUT_DIR_GOLDEN = BASE_DIR / "assay" / "gt_diagnostic_analysis" / "golden"   #changed
OUTPUT_DIR_NEW89 = BASE_DIR / "assay" / "gt_diagnostic_analysis" / "new_89"    #changed
OUTPUT_DIR_GOLDEN.mkdir(parents=True, exist_ok=True)                           #changed
OUTPUT_DIR_NEW89.mkdir(parents=True, exist_ok=True)                            #changed

# API Key
ANTHROPIC_API_KEY = os.environ.get("CLAUDE_API_KEY", "")
if not ANTHROPIC_API_KEY:
    from google.colab import userdata
    ANTHROPIC_API_KEY = userdata.get('CLAUDE_API_KEY')

# Verify paths
print("--- Path Verification ---")
for label, p in [("GT (227)", GT_DIR_227), ("XML (227)", XML_DIR_227),
                  ("GT (golden)", GT_DIR_GOLDEN), ("XML (golden)", XML_DIR_GOLDEN),
                  ("Splits v2", SPLITS_FILE)]:
    exists = p.exists()
    status = "OK" if exists else "MISSING"
    print(f"  {label}: {status} -> {p}")

--- Path Verification ---
  GT (227): OK -> /content/drive/MyDrive/AI6129/ground_truth
  XML (227): OK -> /content/drive/MyDrive/AI6129/xml
  GT (golden): OK -> /content/drive/MyDrive/AI6129/ground_truth/golden
  XML (golden): OK -> /content/drive/MyDrive/AI6129/xml/golden
  Splits v2: OK -> /content/drive/MyDrive/AI6129/assay/validation_splits/assay_tadp_gepa_optimised_splits_v2.json


In [50]:
# Initialize LM
extraction_lm = dspy.LM(
    model="anthropic/claude-haiku-4-5-20251001",
    api_key=ANTHROPIC_API_KEY,
    max_tokens=64000
)
dspy.configure(lm=extraction_lm)
print(f"LM configured: {extraction_lm}")

LM configured: <dspy.clients.lm.LM object at 0x7af6e5a50980>


In [46]:
# =============================================================================
# Load splits v2 and build PMCID lists for both sets
# =============================================================================

with open(SPLITS_FILE, 'r') as f:
    splits_data = json.load(f)

# --- New 89: validation + split_30 ---
validation_pmcids = splits_data['validation_set']
split_30_pmcids = splits_data['split_indices']['split_30']
new_89_pmcids = sorted(set(validation_pmcids + split_30_pmcids))

# --- Golden: list GT files in golden directory ---                            #changed
golden_pmcids = sorted([                                                       #changed
    f.stem for f in GT_DIR_GOLDEN.glob("PMC*.json")                            #changed
])                                                                             #changed

# --- Also extract the 26 replacement PMCIDs for targeted reporting ---        #changed
excluded_records = splits_data.get('excluded_records', {})                      #changed
set_aside = splits_data.get('set_aside_supplementary', {})                     #changed
change_history = splits_data['metadata'].get('change_history', [])             #changed

replacement_pmcids = set()                                                     #changed
for change in change_history:                                                  #changed
    if change.get('decision') == 'DD-2026-006':                                #changed
        replacement_pmcids = set(change.get('replacements_26', []))            #changed
        break                                                                  #changed

surviving_pmcids = set(new_89_pmcids) - replacement_pmcids                     #changed

print(f"--- PMCID Lists ---")
print(f"  Golden records:       {len(golden_pmcids)}")
print(f"  New 89 (V + S30):     {len(new_89_pmcids)}")
print(f"    Validation:         {len(validation_pmcids)}")
print(f"    Split 30:           {len(split_30_pmcids)}")
print(f"    Surviving from v1:  {len(surviving_pmcids)}")
print(f"    New replacements:   {len(replacement_pmcids)}")

--- PMCID Lists ---
  Golden records:       45
  New 89 (V + S30):     89
    Validation:         35
    Split 30:           54
    Surviving from v1:  63
    New replacements:   26


In [47]:
# =============================================================================
# Build XML mappings for both directories
# =============================================================================

def build_xml_mapping(xml_dir: Path) -> Dict[str, str]:
    mapping = {}
    for xml_file in xml_dir.glob("*.xml"):
        pmcid = xml_file.stem.split('_')[0]
        mapping[pmcid] = str(xml_file)
    return mapping

xml_mapping_227 = build_xml_mapping(XML_DIR_227)
xml_mapping_golden = build_xml_mapping(XML_DIR_GOLDEN)                         #changed

print(f"XML files mapped (227 dir): {len(xml_mapping_227)}")
print(f"XML files mapped (golden dir): {len(xml_mapping_golden)}")

# Check coverage                                                               #changed
missing_89 = [p for p in new_89_pmcids if p not in xml_mapping_227]            #changed
missing_golden = [p for p in golden_pmcids if p not in xml_mapping_golden]     #changed

if missing_89:                                                                 #changed
    print(f"  WARNING: {len(missing_89)} new-89 PMCIDs missing XML: {missing_89[:5]}")
if missing_golden:                                                             #changed
    print(f"  WARNING: {len(missing_golden)} golden PMCIDs missing XML: {missing_golden[:5]}")
if not missing_89 and not missing_golden:                                      #changed
    print("  All PMCIDs have XML coverage")

XML files mapped (227 dir): 232
XML files mapped (golden dir): 45
  All PMCIDs have XML coverage


In [48]:
# =============================================================================
# Helper functions
# =============================================================================

def load_ground_truth(pmcid: str, gt_dir: Path) -> Optional[Dict]:             #changed: gt_dir is now a parameter
    gt_path = gt_dir / f"{pmcid}.json"
    if not gt_path.exists():
        return None
    with open(gt_path, 'r', encoding='utf-8') as f:
        return json.load(f)

def extract_text_from_xml(xml_path: str, max_chars: int = 100000) -> str:
    with open(xml_path, 'r', encoding='utf-8') as f:
        xml_content = f.read()
    text = re.sub(r'<[^>]+>', ' ', xml_content)
    text = re.sub(r'\s+', ' ', text).strip()
    if len(text) > max_chars:
        text = text[:int(max_chars*0.7)] + "\n[...TRUNCATED...]\n" + text[-int(max_chars*0.3):]
    return text

def flatten_gt(gt_data: Dict) -> Dict[str, Dict]:
    flattened = {}
    if 'isolates_with_linking' in gt_data:
        iwl = gt_data['isolates_with_linking']
        if isinstance(iwl, dict):
            for iso_id, assays in iwl.items():
                if isinstance(assays, dict):
                    flattened[iso_id] = assays
        elif isinstance(iwl, list):
            for idx, iso in enumerate(iwl):
                if isinstance(iso, dict):
                    iso_id = iso.get('isolate_id', f'IWL_{idx}')
                    flattened[iso_id] = {k:v for k,v in iso.items() if k != 'isolate_id'}
    if 'isolate_without_linking' in gt_data:
        iwol = gt_data['isolate_without_linking']
        if isinstance(iwol, list):
            for idx, item in enumerate(iwol):
                if isinstance(item, dict):
                    flattened[f'UNLINKED_{idx}'] = item
    if 'no_isolates_only_assayinformation' in gt_data:
        nioai = gt_data['no_isolates_only_assayinformation']
        if nioai and isinstance(nioai, dict) and any(nioai.values()):
            flattened['NO_ISOLATE_ID'] = nioai
    return flattened

print("Helper functions defined.")

Helper functions defined.


In [49]:
# =============================================================================
# Pre-flight check: input truncation and output token budget
# Run BEFORE the full analysis to identify documents that may fail
# =============================================================================

def check_truncation(pmcid, xml_map, max_chars=100000):                        #changed
    """Check whether extract_text_from_xml will truncate this document."""
    if pmcid not in xml_map:
        return None

    with open(xml_map[pmcid], 'r', encoding='utf-8') as f:
        xml_content = f.read()

    import re
    text = re.sub(r'<[^>]+>', ' ', xml_content)
    text = re.sub(r'\s+', ' ', text).strip()

    total_len = len(text)
    truncated = total_len > max_chars

    result = {
        'pmcid': pmcid,
        'xml_chars': len(xml_content),
        'text_chars': total_len,
        'truncated': truncated,
        'chars_lost': 0,
    }

    if truncated:
        keep_start = int(max_chars * 0.7)
        keep_end = int(max_chars * 0.3)
        gap_start = keep_start
        gap_end = total_len - keep_end
        result['chars_lost'] = gap_end - gap_start

    return result

def estimate_output_tokens(pmcid, gt_dir):                                     #changed
    """Estimate whether output will exceed max_tokens=8192."""
    gt_data = load_ground_truth(pmcid, gt_dir)
    if gt_data is None:
        return None

    gt_flat = flatten_gt(gt_data)
    total_items = 0
    for iso_id, assays in gt_flat.items():
        for field_name, value in assays.items():
            if value is not None and (not isinstance(value, list) or value):
                total_items += 1

    # Conservative estimate: ~65 tokens per JSON field entry
    est_tokens = total_items * 65
    exceeds = est_tokens > 8192

    return {
        'pmcid': pmcid,
        'isolates': len(gt_flat),
        'gt_items': total_items,
        'est_output_tokens': est_tokens,
        'exceeds_8192': exceeds,
    }

# --- Run checks on golden set ---                                             #changed
print("=" * 70)
print("PRE-FLIGHT: INPUT TRUNCATION CHECK (golden)")
print("=" * 70)
trunc_issues = []
for pmcid in golden_pmcids:
    result = check_truncation(pmcid, xml_mapping_golden)
    if result and result['truncated']:
        trunc_issues.append(result)
        print(f"  TRUNCATED: {pmcid} ({result['text_chars']:,} chars, "
              f"losing {result['chars_lost']:,} chars from middle)")

if not trunc_issues:
    print("  No truncation issues detected in golden set")

print(f"\n{'='*70}")
print("PRE-FLIGHT: OUTPUT TOKEN BUDGET CHECK (golden)")
print("=" * 70)
token_issues = []
for pmcid in golden_pmcids:
    result = estimate_output_tokens(pmcid, GT_DIR_GOLDEN)
    if result and result['exceeds_8192']:
        token_issues.append(result)
        print(f"  EXCEEDS: {pmcid} ({result['isolates']} isolates, "
              f"{result['gt_items']} items, ~{result['est_output_tokens']:,} tokens needed)")

if not token_issues:
    print("  No output token budget issues detected in golden set")

# --- Run checks on new 89 ---                                                #changed
print(f"\n{'='*70}")
print("PRE-FLIGHT: INPUT TRUNCATION CHECK (new 89)")
print("=" * 70)
trunc_89 = []
for pmcid in new_89_pmcids:
    result = check_truncation(pmcid, xml_mapping_227)
    if result and result['truncated']:
        trunc_89.append(result)
        print(f"  TRUNCATED: {pmcid} ({result['text_chars']:,} chars, "
              f"losing {result['chars_lost']:,} chars from middle)")

if not trunc_89:
    print("  No truncation issues detected in new 89 set")

print(f"\n{'='*70}")
print("PRE-FLIGHT: OUTPUT TOKEN BUDGET CHECK (new 89)")
print("=" * 70)
token_89 = []
for pmcid in new_89_pmcids:
    result = estimate_output_tokens(pmcid, GT_DIR_227)
    if result and result['exceeds_8192']:
        token_89.append(result)
        print(f"  EXCEEDS: {pmcid} ({result['isolates']} isolates, "
              f"{result['gt_items']} items, ~{result['est_output_tokens']:,} tokens needed)")

if not token_89:
    print("  No output token budget issues detected in new 89 set")

print(f"\nSummary: {len(trunc_issues)+len(trunc_89)} truncation issues, "
      f"{len(token_issues)+len(token_89)} output budget issues")

PRE-FLIGHT: INPUT TRUNCATION CHECK (golden)
  TRUNCATED: PMC6118388 (121,659 chars, losing 21,659 chars from middle)

PRE-FLIGHT: OUTPUT TOKEN BUDGET CHECK (golden)
  EXCEEDS: PMC4892500 (73 isolates, 287 items, ~18,655 tokens needed)
  EXCEEDS: PMC5033494 (128 isolates, 716 items, ~46,540 tokens needed)
  EXCEEDS: PMC5047355 (669 isolates, 3040 items, ~197,600 tokens needed)
  EXCEEDS: PMC5074519 (55 isolates, 220 items, ~14,300 tokens needed)
  EXCEEDS: PMC5739443 (95 isolates, 556 items, ~36,140 tokens needed)
  EXCEEDS: PMC5799193 (196 isolates, 1374 items, ~89,310 tokens needed)
  EXCEEDS: PMC5841774 (223 isolates, 1115 items, ~72,475 tokens needed)
  EXCEEDS: PMC6026178 (59 isolates, 413 items, ~26,845 tokens needed)
  EXCEEDS: PMC6118388 (88 isolates, 304 items, ~19,760 tokens needed)
  EXCEEDS: PMC6218967 (190 isolates, 1582 items, ~102,830 tokens needed)
  EXCEEDS: PMC6262260 (62 isolates, 240 items, ~15,600 tokens needed)
  EXCEEDS: PMC6412060 (31 isolates, 186 items, ~12,090

In [ ]:
# =============================================================================
# DSPy Signature and Extractor
# NOTE: Signature is deliberately UNCHANGED from v1 to enable fair comparison.
# Definitions file and signature tweaks will be applied in a subsequent step.
# =============================================================================

class AssayExtractionSignature(dspy.Signature):
    """Extract assay information from a PubMed article.

    ASSAY TYPES: serotype, mlst, cgmlst, ast_data, amr, plasmid, virulence_genes, pfge, spi, toxin, phage_type, snp

    OUTPUT FORMAT:
    - Use lowercase field names
    - Return JSON: {isolate_id: {assay_type: value, ...}, ...}
    - Use 'NO_ISOLATE_ID' if no specific isolate identifiers exist
    """
    article_text: str = dspy.InputField(desc="Article text from XML")
    assay_info: str = dspy.OutputField(desc="JSON mapping isolate IDs to assay results")

class AssayExtractor(dspy.Module):
    def __init__(self):
        super().__init__()
        self.predictor = dspy.ChainOfThought(AssayExtractionSignature)

    def forward(self, article_text: str) -> dspy.Prediction:
        return self.predictor(article_text=article_text)

def parse_json_output(raw: str) -> Dict:
    if not raw:
        return {}
    try:
        return json.loads(raw.strip())
    except:
        pass
    match = re.search(r'```(?:json)?\s*([\s\S]*?)```', raw)
    if match:
        try:
            return json.loads(match.group(1).strip())
        except:
            pass
    start = raw.find('{')
    if start >= 0:
        depth = 0
        for i in range(start, len(raw)):
            if raw[i] == '{':
                depth += 1
            elif raw[i] == '}':
                depth -= 1
                if depth == 0:
                    try:
                        return json.loads(raw[start:i+1])
                    except:
                        fixed = re.sub(r',\s*([}\\]])', r'\\1', raw[start:i+1])
                        try:
                            return json.loads(fixed)
                        except:
                            pass
                    break
    return {}

print("DSPy extractor defined (unchanged signature).")

In [ ]:
# =============================================================================
# Normalisation functions
# =============================================================================

def normalise_field(f: str) -> str:
    f = f.lower().strip()
    mappings = {'serovar':'serotype', 'ast':'ast_data', 'amr_genes':'amr',
                'plasmids':'plasmid', 'virulence':'virulence_genes'}
    return mappings.get(f, f)

def normalise_value(v: Any) -> str:
    if v is None:
        return ''
    if isinstance(v, list):
        return '|'.join(sorted([normalise_value(x) for x in v if x]))
    if isinstance(v, dict):
        return json.dumps(v, sort_keys=True).lower()
    s = str(v).lower().strip()
    s = re.sub(r'\s*\([^)]*\)', '', s)
    s = s.replace('-', '').replace('_', '').replace(' ', '')
    return s

def check_value_in_text(value: Any, text: str) -> bool:
    if value is None:
        return False
    text_lower = text.lower()
    if isinstance(value, list):
        return any(check_value_in_text(v, text) for v in value)
    if isinstance(value, dict):
        return any(check_value_in_text(v, text) for v in value.values())
    v_str = str(value).lower()
    if len(v_str) < 2:
        return True
    return v_str in text_lower or v_str.replace('-','').replace('_','') in text_lower.replace('-','').replace('_','')

print("Normalisation functions defined.")

In [ ]:
# =============================================================================
# Diagnostic data structure
# =============================================================================

@dataclass
class DocumentDiagnostic:
    pmcid: str
    set_label: str = ''                                                        #changed: track which set this belongs to
    is_replacement: bool = False                                               #changed: flag 26 replacement records
    gt_isolate_count: int = 0
    gt_isolate_ids: List[str] = field(default_factory=list)
    gt_assay_types: List[str] = field(default_factory=list)
    gt_total_items: int = 0
    ext_isolate_count: int = 0
    ext_isolate_ids: List[str] = field(default_factory=list)
    ext_assay_types: List[str] = field(default_factory=list)
    ext_total_items: int = 0
    id_exact_matches: int = 0
    gt_ids_in_text: int = 0
    gt_ids_not_in_text: int = 0
    gt_id_pattern: str = ''
    ext_id_pattern: str = ''
    issue_category: str = ''
    issue_details: str = ''
    primary_f1: float = 0.0
    optimistic_f1: float = 0.0
    tp: int = 0
    fp: int = 0
    up: int = 0
    fn: int = 0
    sample_tp: List[str] = field(default_factory=list)
    sample_fp: List[str] = field(default_factory=list)
    sample_up: List[str] = field(default_factory=list)
    sample_fn: List[str] = field(default_factory=list)

print("DocumentDiagnostic defined (with set_label, is_replacement).")

In [ ]:
def detect_id_pattern(ids: Set[str]) -> str:
    if not ids or ids == {'NO_ISOLATE_ID'}:
        return 'document_level'
    patterns = []
    for id_ in ids:
        if id_ == 'NO_ISOLATE_ID':
            continue
        il = id_.lower()
        if re.match(r'^[a-z]{2,6}\d{4}[-_]\d{2,3}$', il):
            patterns.append('lab_sequential')
        elif re.match(r'^[a-z]{1,3}\d{1,3}$', il):
            patterns.append('simple_code')
        elif re.match(r'^(samn|samea|mg|cp)\d+', il):
            patterns.append('accession')
        else:
            patterns.append('other')
    return Counter(patterns).most_common(1)[0][0] if patterns else 'unknown'

def classify_issue(diag: DocumentDiagnostic) -> Tuple[str, str]:
    """Classify the issue category based on diagnostic data."""
    # Granularity mismatch: GT has many isolates, extraction has NO_ISOLATE_ID
    if (diag.gt_isolate_count > 3 and
        diag.ext_isolate_ids == ['NO_ISOLATE_ID'] and
        diag.gt_ids_not_in_text > diag.gt_ids_in_text):
        return 'granularity_mismatch', f"GT has {diag.gt_isolate_count} isolates with IDs not in text; extraction uses document-level"

    # Over-extraction: More extracted than GT, low FN
    if diag.fn == 0 and diag.up > 5 and diag.ext_total_items > diag.gt_total_items * 1.5:
        return 'over_extraction', f"Extraction has {diag.ext_total_items} items vs GT {diag.gt_total_items}; {diag.up} UP items"

    # ID scheme mismatch: Both have IDs but different patterns
    if (diag.id_exact_matches == 0 and
        diag.gt_id_pattern != 'document_level' and
        diag.ext_id_pattern != 'document_level' and
        diag.gt_id_pattern != diag.ext_id_pattern):
        return 'id_scheme_mismatch', f"GT uses {diag.gt_id_pattern}, extraction uses {diag.ext_id_pattern}"

    # Format mismatch: Have matches but low F1 due to value format differences
    if diag.id_exact_matches > 0 and diag.primary_f1 < 0.3 and diag.fn > diag.tp:
        return 'format_mismatch', f"ID matches exist but F1={diag.primary_f1:.2f} suggests value format issues"

    # GT errors: GT items not extractable and not in text
    if diag.fn > 0 and diag.gt_ids_not_in_text > diag.gt_ids_in_text:
        return 'gt_data_not_in_article', f"{diag.gt_ids_not_in_text}/{diag.gt_isolate_count} GT IDs not found in article text"

    # Good match
    if diag.primary_f1 >= 0.5:
        return 'acceptable', f"F1={diag.primary_f1:.2f} indicates reasonable extraction quality"

    # Mixed issues
    return 'mixed_issues', f"TP={diag.tp}, FP={diag.fp}, UP={diag.up}, FN={diag.fn}"

print("Classification functions defined.")

In [ ]:
def compare_and_score(gt_flat: Dict, ext: Dict, article_text: str) -> Tuple[int, int, int, int, List, List, List, List]:
    """
    Compare GT and extraction, return TP, FP, UP, FN counts and samples.

    TP: Extracted item matches GT
    FP: Extracted item not in GT and not in article
    UP: Extracted item not in GT but IS in article (valid extraction, GT gap)
    FN: GT item not extracted
    """
    tp_items = []
    fp_items = []
    up_items = []
    fn_items = []

    # Normalise GT
    gt_normalised = {}
    for iso_id, assays in gt_flat.items():
        for field, value in assays.items():
            if value is not None and (not isinstance(value, list) or value):
                nf = normalise_field(field)
                nv = normalise_value(value)
                if nv:
                    key = f"{iso_id}|{nf}"
                    gt_normalised[key] = (iso_id, nf, value, nv)

    # Normalise extraction
    ext_normalised = {}
    for iso_id, assays in ext.items():
        if not isinstance(assays, dict):
            continue
        for field, value in assays.items():
            if value is not None and (not isinstance(value, list) or value):
                nf = normalise_field(field)
                nv = normalise_value(value)
                if nv:
                    key = f"{iso_id}|{nf}"
                    ext_normalised[key] = (iso_id, nf, value, nv)

    # Build value-based lookup for lenient matching
    gt_by_field_value = {}
    for key, (iso_id, nf, orig_v, nv) in gt_normalised.items():
        fv_key = f"{nf}|{nv}"
        gt_by_field_value[fv_key] = key

    matched_gt = set()
    matched_ext = set()

    # Check extractions
    for ext_key, (iso_id, nf, orig_v, nv) in ext_normalised.items():
        # Exact match (same ID and field)
        if ext_key in gt_normalised and gt_normalised[ext_key][3] == nv:
            tp_items.append(f"{ext_key}={str(orig_v)[:50]}")
            matched_gt.add(ext_key)
            matched_ext.add(ext_key)
            continue

        # Value-based match (different ID, same field+value)
        fv_key = f"{nf}|{nv}"
        if fv_key in gt_by_field_value:
            gt_key = gt_by_field_value[fv_key]
            if gt_key not in matched_gt:
                tp_items.append(f"{ext_key}={str(orig_v)[:50]} (value-matched to {gt_key})")
                matched_gt.add(gt_key)
                matched_ext.add(ext_key)
                continue

        # Not in GT - check if in article (UP vs FP)
        if check_value_in_text(orig_v, article_text):
            up_items.append(f"{ext_key}={str(orig_v)[:50]}")
        else:
            fp_items.append(f"{ext_key}={str(orig_v)[:50]}")

    # Find FN (GT items not matched)
    for gt_key, (iso_id, nf, orig_v, nv) in gt_normalised.items():
        if gt_key not in matched_gt:
            fn_items.append(f"{gt_key}={str(orig_v)[:50]}")

    return (len(tp_items), len(fp_items), len(up_items), len(fn_items),
            tp_items[:5], fp_items[:5], up_items[:5], fn_items[:5])

print("Comparison function defined.")

In [ ]:
def analyse_document(pmcid: str, extractor: AssayExtractor,
                    gt_dir: Path, xml_map: Dict[str, str],                     #changed: parameterised
                    set_label: str = '',                                        #changed
                    is_replacement: bool = False                                #changed
                    ) -> Optional[DocumentDiagnostic]:
    """Run full diagnostic analysis on a single document."""
    diag = DocumentDiagnostic(pmcid=pmcid, set_label=set_label,
                              is_replacement=is_replacement)                    #changed

    # Load GT from the specified directory                                     #changed
    gt_data = load_ground_truth(pmcid, gt_dir)                                 #changed
    if gt_data is None:
        return None

    # Load article from the specified XML mapping                              #changed
    if pmcid not in xml_map:                                                   #changed
        return None
    article_text = extract_text_from_xml(xml_map[pmcid])                       #changed

    # Flatten GT
    gt_flat = flatten_gt(gt_data)
    diag.gt_isolate_count = len(gt_flat)
    diag.gt_isolate_ids = list(gt_flat.keys())[:20]

    # Count GT items and assay types
    gt_assay_types = set()
    gt_item_count = 0
    for iso_id, assays in gt_flat.items():
        for field_name, value in assays.items():
            if value is not None and (not isinstance(value, list) or value):
                gt_assay_types.add(normalise_field(field_name))
                gt_item_count += 1
    diag.gt_assay_types = sorted(gt_assay_types)
    diag.gt_total_items = gt_item_count

    # Check GT IDs in text
    for iso_id in gt_flat.keys():
        if iso_id == 'NO_ISOLATE_ID':
            diag.gt_ids_in_text += 1
        elif iso_id.lower() in article_text.lower():
            diag.gt_ids_in_text += 1
        else:
            diag.gt_ids_not_in_text += 1

    diag.gt_id_pattern = detect_id_pattern(set(gt_flat.keys()))

    # Run extraction
    try:
        prediction = extractor(article_text=article_text)
        ext = parse_json_output(prediction.assay_info)
    except Exception as e:
        print(f"  Extraction error: {e}")
        ext = {}

    # Analyse extraction
    diag.ext_isolate_count = len(ext)
    diag.ext_isolate_ids = list(ext.keys())[:20]

    ext_assay_types = set()
    ext_item_count = 0
    for iso_id, assays in ext.items():
        if isinstance(assays, dict):
            for field_name, value in assays.items():
                if value is not None:
                    ext_assay_types.add(normalise_field(field_name))
                    ext_item_count += 1
    diag.ext_assay_types = sorted(ext_assay_types)
    diag.ext_total_items = ext_item_count
    diag.ext_id_pattern = detect_id_pattern(set(ext.keys()))

    # ID matching
    diag.id_exact_matches = len(set(gt_flat.keys()) & set(ext.keys()))

    # Compare and score
    tp, fp, up, fn, s_tp, s_fp, s_up, s_fn = compare_and_score(gt_flat, ext, article_text)
    diag.tp = tp
    diag.fp = fp
    diag.up = up
    diag.fn = fn
    diag.sample_tp = s_tp
    diag.sample_fp = s_fp
    diag.sample_up = s_up
    diag.sample_fn = s_fn

    # Primary F1: UP excluded
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    diag.primary_f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    # Optimistic F1: UP counted as TP
    opt_precision = (tp + up) / (tp + fp + up) if (tp + fp + up) > 0 else 0
    opt_recall = (tp + up) / (tp + up + fn) if (tp + up + fn) > 0 else 0
    diag.optimistic_f1 = 2 * opt_precision * opt_recall / (opt_precision + opt_recall) if (opt_precision + opt_recall) > 0 else 0

    # Classify issue
    diag.issue_category, diag.issue_details = classify_issue(diag)

    return diag

print("Document analysis function defined (parameterised for GT dir and XML mapping).")

## 5. Run Diagnostic Analysis

**Execution order:**
1. Golden test set first (45 records) — blocking dependency
2. New 89 diagnostic set second — baseline for reconstituted splits

Each set uses its own GT directory and XML mapping.

In [ ]:
# =============================================================================
# Quick test: 3 from golden + 3 from new 89
# =============================================================================

extractor = AssayExtractor()

# Test golden                                                                  #changed
TEST_GOLDEN = golden_pmcids[:3]                                                #changed
print(f"--- Testing golden ({len(TEST_GOLDEN)} docs) ---")
golden_test_results = []
for i, pmcid in enumerate(TEST_GOLDEN):
    print(f"  [{i+1}] {pmcid}")
    diag = analyse_document(pmcid, extractor,
                            gt_dir=GT_DIR_GOLDEN,                              #changed
                            xml_map=xml_mapping_golden,                        #changed
                            set_label='golden')                                #changed
    if diag:
        golden_test_results.append(diag)
        print(f"    GT: {diag.gt_isolate_count} isolates, {diag.gt_total_items} items")
        print(f"    Ext: {diag.ext_isolate_count} isolates, {diag.ext_total_items} items")
        print(f"    TP={diag.tp}, FP={diag.fp}, UP={diag.up}, FN={diag.fn}")
        print(f"    Primary F1={diag.primary_f1:.3f}, Issue: {diag.issue_category}")
    else:
        print(f"    SKIPPED (missing GT or XML)")
    time.sleep(1)

# Test new 89                                                                  #changed
TEST_NEW89 = new_89_pmcids[:3]                                                 #changed
print(f"\n--- Testing new 89 ({len(TEST_NEW89)} docs) ---")
new89_test_results = []
for i, pmcid in enumerate(TEST_NEW89):
    print(f"  [{i+1}] {pmcid}")
    diag = analyse_document(pmcid, extractor,
                            gt_dir=GT_DIR_227,                                 #changed
                            xml_map=xml_mapping_227,                           #changed
                            set_label='new_89',                                #changed
                            is_replacement=(pmcid in replacement_pmcids))      #changed
    if diag:
        new89_test_results.append(diag)
        repl_tag = " [REPLACEMENT]" if diag.is_replacement else ""
        print(f"    GT: {diag.gt_isolate_count} isolates, {diag.gt_total_items} items{repl_tag}")
        print(f"    Ext: {diag.ext_isolate_count} isolates, {diag.ext_total_items} items")
        print(f"    TP={diag.tp}, FP={diag.fp}, UP={diag.up}, FN={diag.fn}")
        print(f"    Primary F1={diag.primary_f1:.3f}, Issue: {diag.issue_category}")
    else:
        print(f"    SKIPPED (missing GT or XML)")
    time.sleep(1)

print(f"\nTest complete: {len(golden_test_results)} golden + {len(new89_test_results)} new-89")

In [ ]:
# =============================================================================
# Full analysis — Golden first, then New 89
# WARNING: ~134 API calls total (~45 golden + ~89 new). Approx 30-45 minutes.
# =============================================================================

RUN_FULL_ANALYSIS = True  # Change to True to run

if RUN_FULL_ANALYSIS:
    extractor = AssayExtractor()

    # ---- PHASE 1: Golden test set (blocking dependency) ----                 #changed
    golden_results = []
    print(f"{'='*70}")
    print(f"PHASE 1: Golden test set ({len(golden_pmcids)} documents)")
    print(f"{'='*70}")

    for i, pmcid in enumerate(golden_pmcids):
        if (i + 1) % 10 == 0:
            print(f"  Progress: {i+1}/{len(golden_pmcids)}")

        diag = analyse_document(pmcid, extractor,
                                gt_dir=GT_DIR_GOLDEN,
                                xml_map=xml_mapping_golden,
                                set_label='golden')
        if diag:
            golden_results.append(diag)
        else:
            print(f"  SKIPPED: {pmcid} (missing GT or XML)")

        time.sleep(1.5)

    print(f"Phase 1 complete: {len(golden_results)}/{len(golden_pmcids)} documents analysed")

    # Save golden results immediately (in case phase 2 fails)                  #changed
    golden_data = [asdict(d) for d in golden_results]
    with open(OUTPUT_DIR_GOLDEN / 'diagnostic_results.json', 'w') as f:
        json.dump(golden_data, f, indent=2, default=str)
    print(f"Golden results saved to {OUTPUT_DIR_GOLDEN / 'diagnostic_results.json'}")

    # ---- PHASE 2: New 89 diagnostic set ----                                 #changed
    new89_results = []
    print(f"\n{'='*70}")
    print(f"PHASE 2: New 89 diagnostic set ({len(new_89_pmcids)} documents)")
    print(f"{'='*70}")

    for i, pmcid in enumerate(new_89_pmcids):
        if (i + 1) % 10 == 0:
            print(f"  Progress: {i+1}/{len(new_89_pmcids)}")

        diag = analyse_document(pmcid, extractor,
                                gt_dir=GT_DIR_227,
                                xml_map=xml_mapping_227,
                                set_label='new_89',
                                is_replacement=(pmcid in replacement_pmcids))
        if diag:
            new89_results.append(diag)
        else:
            print(f"  SKIPPED: {pmcid} (missing GT or XML)")

        time.sleep(1.5)

    print(f"Phase 2 complete: {len(new89_results)}/{len(new_89_pmcids)} documents analysed")

    # Save new 89 results                                                      #changed
    new89_data = [asdict(d) for d in new89_results]
    with open(OUTPUT_DIR_NEW89 / 'diagnostic_results.json', 'w') as f:
        json.dump(new89_data, f, indent=2, default=str)
    print(f"New 89 results saved to {OUTPUT_DIR_NEW89 / 'diagnostic_results.json'}")

else:
    print("Set RUN_FULL_ANALYSIS = True to run full analysis")
    golden_results = golden_test_results
    new89_results = new89_test_results

## 6. Analysis and Reporting

Reports generated separately for each set, then a combined comparison.

In [ ]:
# =============================================================================
# Summary report generator (parameterised by set label)
# =============================================================================

def generate_summary_report(results: List[DocumentDiagnostic],
                            label: str = ""):                                  #changed
    """Generate summary report for a set of diagnostic results."""
    if not results:
        print(f"No results for {label}")
        return

    print(f"{'='*70}")
    print(f"GT DIAGNOSTIC SUMMARY: {label.upper()}")
    print(f"{'='*70}")
    print(f"Total documents analysed: {len(results)}")

    # Replacement breakdown (for new 89 only)                                  #changed
    replacements = [d for d in results if d.is_replacement]                    #changed
    survivors = [d for d in results if not d.is_replacement]                   #changed
    if replacements:                                                           #changed
        print(f"  Surviving from v1: {len(survivors)}")                        #changed
        print(f"  New replacements:  {len(replacements)}")                     #changed

    # Issue category breakdown
    categories = Counter(d.issue_category for d in results)
    print(f"\n--- ISSUE CATEGORY BREAKDOWN ---")
    for cat, count in categories.most_common():
        pct = count / len(results) * 100
        print(f"  {cat}: {count} ({pct:.1f}%)")

    # Replacement-specific breakdown                                           #changed
    if replacements:                                                           #changed
        print(f"\n--- REPLACEMENT RECORDS ISSUE BREAKDOWN ---")               #changed
        rep_cats = Counter(d.issue_category for d in replacements)             #changed
        for cat, count in rep_cats.most_common():                              #changed
            print(f"  {cat}: {count}")                                         #changed

    # F1 score distribution
    print(f"\n--- F1 SCORE DISTRIBUTION ---")
    f1_zero = sum(1 for d in results if d.primary_f1 == 0)
    f1_low = sum(1 for d in results if 0 < d.primary_f1 < 0.5)
    f1_med = sum(1 for d in results if 0.5 <= d.primary_f1 < 0.8)
    f1_high = sum(1 for d in results if d.primary_f1 >= 0.8)
    print(f"  F1 = 0: {f1_zero} ({f1_zero/len(results)*100:.1f}%)")
    print(f"  0 < F1 < 0.5: {f1_low} ({f1_low/len(results)*100:.1f}%)")
    print(f"  0.5 <= F1 < 0.8: {f1_med} ({f1_med/len(results)*100:.1f}%)")
    print(f"  F1 >= 0.8: {f1_high} ({f1_high/len(results)*100:.1f}%)")

    avg_primary = sum(d.primary_f1 for d in results) / len(results)
    avg_optimistic = sum(d.optimistic_f1 for d in results) / len(results)
    print(f"\n  Mean Primary F1: {avg_primary:.4f}")
    print(f"  Mean Optimistic F1: {avg_optimistic:.4f}")
    print(f"  Difference: +{avg_optimistic - avg_primary:.4f}")

    # ID pattern analysis
    print(f"\n--- ID PATTERN ANALYSIS ---")
    gt_patterns = Counter(d.gt_id_pattern for d in results)
    ext_patterns = Counter(d.ext_id_pattern for d in results)
    print(f"GT patterns:")
    for pat, count in gt_patterns.most_common():
        print(f"  {pat}: {count}")
    print(f"Extraction patterns:")
    for pat, count in ext_patterns.most_common():
        print(f"  {pat}: {count}")

    # GT IDs in text
    total_gt_ids = sum(d.gt_isolate_count for d in results)
    total_in_text = sum(d.gt_ids_in_text for d in results)
    total_not_in_text = sum(d.gt_ids_not_in_text for d in results)
    print(f"\n--- GT IDs IN ARTICLE TEXT ---")
    print(f"  Total GT isolate IDs: {total_gt_ids}")
    if total_gt_ids > 0:
        print(f"  Found in text: {total_in_text} ({total_in_text/total_gt_ids*100:.1f}%)")
        print(f"  NOT in text: {total_not_in_text} ({total_not_in_text/total_gt_ids*100:.1f}%)")

    # Aggregated counts
    total_tp = sum(d.tp for d in results)
    total_fp = sum(d.fp for d in results)
    total_up = sum(d.up for d in results)
    total_fn = sum(d.fn for d in results)
    print(f"\n--- AGGREGATED COUNTS ---")
    print(f"  Total TP: {total_tp}")
    print(f"  Total FP: {total_fp}")
    print(f"  Total UP: {total_up}")
    print(f"  Total FN: {total_fn}")

# Run reports for both sets
generate_summary_report(golden_results, "Golden Test Set (45)")
print("\n")
generate_summary_report(new89_results, "New 89 Diagnostic Set")

In [ ]:
def generate_detailed_table(results: List[DocumentDiagnostic],
                           label: str = ""):                                   #changed
    """Generate detailed per-document table."""
    print(f"\n{'='*130}")
    print(f"DETAILED BREAKDOWN: {label.upper()}")
    print(f"{'='*130}")
    print(f"{'PMCID':<15} {'Repl':>4} {'Pri_F1':>7} {'Opt_F1':>7} {'TP':>4} {'FP':>4} {'UP':>4} {'FN':>4} {'GT_IDs':>7} {'In_Txt':>7} {'Issue_Category':<25}")  #changed
    print("-"*130)

    for d in sorted(results, key=lambda x: x.primary_f1):
        repl_flag = "YES" if d.is_replacement else ""                          #changed
        print(f"{d.pmcid:<15} {repl_flag:>4} {d.primary_f1:>7.3f} {d.optimistic_f1:>7.3f} {d.tp:>4} {d.fp:>4} {d.up:>4} {d.fn:>4} {d.gt_isolate_count:>7} {d.gt_ids_in_text:>7} {d.issue_category:<25}")

generate_detailed_table(golden_results, "Golden Test Set")
generate_detailed_table(new89_results, "New 89 Diagnostic Set")

In [ ]:
def show_issue_examples(results: List[DocumentDiagnostic], category: str, n: int = 3):
    """Show example documents for a specific issue category."""
    examples = [d for d in results if d.issue_category == category][:n]
    if not examples:
        return

    print(f"\n{'='*70}")
    print(f"EXAMPLES: {category.upper()} [{examples[0].set_label}]")           #changed
    print(f"{'='*70}")

    for d in examples:
        repl_tag = " [REPLACEMENT]" if d.is_replacement else ""                #changed
        print(f"\n--- {d.pmcid}{repl_tag} ---")
        print(f"Issue: {d.issue_details}")
        print(f"GT: {d.gt_isolate_count} isolates ({d.gt_id_pattern}), {d.gt_total_items} items")
        print(f"Ext: {d.ext_isolate_count} isolates ({d.ext_id_pattern}), {d.ext_total_items} items")
        print(f"GT IDs in text: {d.gt_ids_in_text}/{d.gt_isolate_count}")
        print(f"Primary F1: {d.primary_f1:.3f}, Optimistic F1: {d.optimistic_f1:.3f}")
        print(f"\nSample GT IDs: {d.gt_isolate_ids[:5]}")
        print(f"Sample Ext IDs: {d.ext_isolate_ids[:5]}")
        if d.sample_fn:
            print(f"Sample FN: {d.sample_fn[:3]}")
        if d.sample_up:
            print(f"Sample UP: {d.sample_up[:3]}")

# Show examples per category per set                                           #changed
print("\n" + "#"*70)
print("# GOLDEN SET ISSUE EXAMPLES")
print("#"*70)
golden_cats = set(d.issue_category for d in golden_results)
for cat in golden_cats:
    show_issue_examples(golden_results, cat, n=2)

print("\n" + "#"*70)
print("# NEW 89 ISSUE EXAMPLES")
print("#"*70)
new89_cats = set(d.issue_category for d in new89_results)
for cat in new89_cats:
    show_issue_examples(new89_results, cat, n=2)

In [ ]:
# =============================================================================
# Cross-set comparison: Golden vs New 89
# =============================================================================

def compare_sets(golden: List[DocumentDiagnostic],                             #changed
                 new89: List[DocumentDiagnostic]):                              #changed
    """Side-by-side comparison of the two record sets."""
    print(f"{'='*70}")
    print(f"CROSS-SET COMPARISON: Golden vs New 89")
    print(f"{'='*70}")

    def stats(results):
        if not results:
            return {}
        n = len(results)
        return {
            'n': n,
            'mean_f1': sum(d.primary_f1 for d in results) / n,
            'mean_opt_f1': sum(d.optimistic_f1 for d in results) / n,
            'f1_zero_pct': sum(1 for d in results if d.primary_f1 == 0) / n * 100,
            'f1_high_pct': sum(1 for d in results if d.primary_f1 >= 0.5) / n * 100,
            'ids_in_text_pct': (sum(d.gt_ids_in_text for d in results) /
                                max(sum(d.gt_isolate_count for d in results), 1) * 100),
            'categories': dict(Counter(d.issue_category for d in results).most_common()),
        }

    g = stats(golden)
    n = stats(new89)

    # Also compute stats for replacement subset                                #changed
    replacements = [d for d in new89 if d.is_replacement]                      #changed
    survivors = [d for d in new89 if not d.is_replacement]                     #changed
    r = stats(replacements) if replacements else {}                            #changed
    s = stats(survivors) if survivors else {}                                  #changed

    header = f"{'Metric':<30} {'Golden':>12} {'New 89':>12} {'Surviving':>12} {'Replace':>12}"
    print(header)
    print("-" * len(header))

    def row(label, key, fmt='.3f'):
        vals = [g.get(key, '-'), n.get(key, '-'), s.get(key, '-'), r.get(key, '-')]
        parts = []
        for v in vals:
            if isinstance(v, float):
                parts.append(f"{v:{fmt}}")
            elif isinstance(v, int):
                parts.append(f"{v}")
            else:
                parts.append(f"{v}")
        print(f"{label:<30} {parts[0]:>12} {parts[1]:>12} {parts[2]:>12} {parts[3]:>12}")

    row("Documents", 'n', 'd')
    row("Mean Primary F1", 'mean_f1')
    row("Mean Optimistic F1", 'mean_opt_f1')
    row("F1 = 0 (%)", 'f1_zero_pct', '.1f')
    row("F1 >= 0.5 (%)", 'f1_high_pct', '.1f')
    row("GT IDs in text (%)", 'ids_in_text_pct', '.1f')

    # Category comparison
    all_cats = sorted(set(list(g.get('categories', {}).keys()) +
                          list(n.get('categories', {}).keys())))
    print(f"\n{'Issue Category':<30} {'Golden':>12} {'New 89':>12} {'Surviving':>12} {'Replace':>12}")
    print("-" * 78)
    for cat in all_cats:
        gv = g.get('categories', {}).get(cat, 0)
        nv = n.get('categories', {}).get(cat, 0)
        sv = s.get('categories', {}).get(cat, 0)
        rv = r.get('categories', {}).get(cat, 0)
        print(f"  {cat:<28} {gv:>12} {nv:>12} {sv:>12} {rv:>12}")

    # Flag golden records that may need attention                              #changed
    golden_issues = [d for d in golden if d.issue_category in                  #changed
                     ('granularity_mismatch', 'gt_data_not_in_article',        #changed
                      'id_scheme_mismatch')]                                   #changed
    if golden_issues:                                                          #changed
        print(f"\n--- GOLDEN RECORDS REQUIRING ATTENTION ({len(golden_issues)}) ---")
        for d in sorted(golden_issues, key=lambda x: x.primary_f1):           #changed
            print(f"  {d.pmcid}: {d.issue_category} (F1={d.primary_f1:.3f}, "
                  f"GT IDs in text: {d.gt_ids_in_text}/{d.gt_isolate_count})")
    else:                                                                      #changed
        print(f"\nNo structural issues detected in golden records.")

compare_sets(golden_results, new89_results)

In [ ]:
# =============================================================================
# Save comprehensive reports for both sets
# =============================================================================

def save_report(results: List[DocumentDiagnostic], output_dir: Path,
                label: str = ""):                                              #changed
    """Save comprehensive diagnostic report."""
    if not results:
        print(f"No results to save for {label}")
        return

    report = {
        'metadata': {
            'generated_at': datetime.now().isoformat(),
            'set_label': label,                                                #changed
            'total_documents': len(results),
            'replacement_count': sum(1 for d in results if d.is_replacement),  #changed
            'splits_file': str(SPLITS_FILE),                                   #changed
        },
        'summary': {
            'issue_categories': dict(Counter(d.issue_category for d in results)),
            'mean_primary_f1': sum(d.primary_f1 for d in results) / len(results),
            'mean_optimistic_f1': sum(d.optimistic_f1 for d in results) / len(results),
            'f1_zero_count': sum(1 for d in results if d.primary_f1 == 0),
            'gt_patterns': dict(Counter(d.gt_id_pattern for d in results)),
            'ext_patterns': dict(Counter(d.ext_id_pattern for d in results)),
            'total_gt_ids': sum(d.gt_isolate_count for d in results),
            'gt_ids_in_text': sum(d.gt_ids_in_text for d in results),
            'gt_ids_not_in_text': sum(d.gt_ids_not_in_text for d in results),
        },
        'documents': [asdict(d) for d in results]
    }

    with open(output_dir / 'gt_diagnostic_report.json', 'w') as f:
        json.dump(report, f, indent=2, default=str)

    # CSV for easy viewing
    csv_lines = ['PMCID,Set_Label,Is_Replacement,Primary_F1,Optimistic_F1,TP,FP,UP,FN,GT_Isolates,GT_IDs_In_Text,Issue_Category']  #changed
    for d in sorted(results, key=lambda x: x.primary_f1):
        csv_lines.append(f"{d.pmcid},{d.set_label},{d.is_replacement},{d.primary_f1:.4f},{d.optimistic_f1:.4f},{d.tp},{d.fp},{d.up},{d.fn},{d.gt_isolate_count},{d.gt_ids_in_text},{d.issue_category}")

    with open(output_dir / 'gt_diagnostic_summary.csv', 'w') as f:
        f.write('\n'.join(csv_lines))

    print(f"  {label} reports saved to {output_dir}")

save_report(golden_results, OUTPUT_DIR_GOLDEN, "golden")
save_report(new89_results, OUTPUT_DIR_NEW89, "new_89")

# Also save a combined CSV for quick cross-set analysis                        #changed
combined_dir = BASE_DIR / "assay" / "gt_diagnostic_analysis"                   #changed
all_combined = golden_results + new89_results                                  #changed
csv_lines = ['PMCID,Set_Label,Is_Replacement,Primary_F1,Optimistic_F1,TP,FP,UP,FN,GT_Isolates,GT_IDs_In_Text,Issue_Category']
for d in sorted(all_combined, key=lambda x: (x.set_label, x.primary_f1)):
    csv_lines.append(f"{d.pmcid},{d.set_label},{d.is_replacement},{d.primary_f1:.4f},{d.optimistic_f1:.4f},{d.tp},{d.fp},{d.up},{d.fn},{d.gt_isolate_count},{d.gt_ids_in_text},{d.issue_category}")
with open(combined_dir / 'gt_diagnostic_combined.csv', 'w') as f:
    f.write('\n'.join(csv_lines))
print(f"  Combined CSV saved to {combined_dir / 'gt_diagnostic_combined.csv'}")

## 7. Conclusions

### Key Questions This Notebook Answers:

1. **Are the 45 golden records structurally sound?**
   - Check golden issue categories for `granularity_mismatch`, `gt_data_not_in_article`, `id_scheme_mismatch`
   - Any golden records in these categories need manual review BEFORE running GEPA experiments

2. **Are the 26 replacement records clean?**
   - Filter new 89 results by `is_replacement == True` and check for the same issues
   - Compare replacement F1 distribution against surviving records

3. **Is the reconstituted 89-set comparable to the original?**
   - Compare cross-set metrics: mean F1, issue category distribution, GT IDs in text %
   - Replacement records should not disproportionately cluster in problem categories

4. **What is the baseline performance before signature tweaks?**
   - The unchanged signature provides the control condition
   - Next step: add definitions file and re-run to measure improvement delta

### Decision Points After This Run:

- If golden has > 5 problem records: **stop and fix golden GT before proceeding**
- If golden is clean but replacements have issues: **flag specific replacement PMCIDs for review**
- If both sets are clean: **proceed to signature tweaks and GEPA optimisation**